# Export smartmeter data from netzOÖ

Download data from eservice.netzooe.at consumption data:

1. Download der 15-Minutenwerte über das NetzOÖ e-Service-Portal
2. Vertrag auswählen
3. Verbrauchsübersicht
4. Zeitraum auswählen
5. Ausgewählte Daten herunterladen

and then download feed-in data the same way.


## Parameter

In [ ]:
#Add downloadet Files
consumption_file = "Managerdaten_AT00.csv"
feedin_file = "Managerdaten_AT000000.csv"

## Convert Data

In [ ]:
import pandas as pd

def load_and_process_data(consumption_file, feedin_file):
    consumption_df = pd.read_csv(consumption_file, delimiter=';')
    feedin_df = pd.read_csv(feedin_file, delimiter=';')
    consumption_df = consumption_df.dropna(axis=1, how='all')
    feedin_df = feedin_df.dropna(axis=1, how='all')
    consumption_df.columns = ['timestamp', 'kWh', 'kW', 'status']
    feedin_df.columns = ['timestamp', 'kWh', 'kW', 'status']
    consumption_df['timestamp'] = pd.to_datetime(consumption_df['timestamp'], format='%d.%m.%Y %H:%M')
    feedin_df['timestamp'] = pd.to_datetime(feedin_df['timestamp'], format='%d.%m.%Y %H:%M')
    consumption_df = consumption_df[consumption_df['status'] == 'VALID']
    feedin_df = feedin_df[feedin_df['status'] == 'VALID']
    consumption_df['kWh'] = consumption_df['kWh'].str.replace(',', '.').astype(float)
    consumption_df['kW'] = consumption_df['kW'].str.replace(',', '.').astype(float)
    feedin_df['kWh'] = feedin_df['kWh'].str.replace(',', '.').astype(float)
    feedin_df['kW'] = feedin_df['kW'].str.replace(',', '.').astype(float)
    df = pd.merge(consumption_df[['timestamp', 'kW']], feedin_df[['timestamp', 'kW']], on="timestamp", how="outer", suffixes=('_cons', '_feed'))
    df['power'] = (df['kW_cons'] - df['kW_feed']) * 1000
    df = df.rename(columns={'kW_cons': 'consumption', 'kW_feed': 'feedin'})
    df = df[['timestamp', 'power', 'consumption', 'feedin']]
    df = df.dropna(subset=['power', 'consumption', 'feedin'])
    df = df.sort_values(by='timestamp').reset_index(drop=True)

    return df

df = load_and_process_data(consumption_file, feedin_file)

# Print 
print(df.head())

# Save the processed data 
df = df[['timestamp', 'power']]  # Keep only the 'timestamp' and 'power' columns
df.to_csv("../../sample_data.csv", index=False)


            timestamp  power  consumption  feedin
0 2025-03-22 00:00:00  240.0        0.240     0.0
1 2025-03-22 00:15:00  252.0        0.252     0.0
2 2025-03-22 00:30:00  236.0        0.236     0.0
3 2025-03-22 00:45:00  272.0        0.272     0.0
4 2025-03-22 01:00:00  260.0        0.260     0.0
